In [1]:
# ✱✱✱  RUN THIS CELL  ✱✱✱
import glob, os, math, numpy as np, soundfile as sf, librosa, torch
import torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from tqdm import tqdm
from gammatone.filters import make_erb_filters, erb_space, erb_filterbank

# ----------------- hyper-params -----------------
AUDIO_DIR   = "./audio"        # folder of .wav files
SR          = 16_000
N_FILT      = 256              # ERB channels
LP_FC       = 770              # 1-pole low-pass Hz
BATCH       = 1                # SGD batch
EPOCHS      = 40
LR          = 3e-4
LAMBDA_L1   = 1e-3             # sparsity weight
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
# ------------------------------------------------

# ---------- gammatone impulse responses ----------
def gfb_impulses(sr=SR, n_filters=N_FILT, low_freq=50.):
    cf = erb_space(low_freq=low_freq, high_freq=sr/2, num=n_filters)
    ir  = erb_filterbank(np.pad([1.0], (0,511)), make_erb_filters(sr, cf))
    return torch.from_numpy(ir).unsqueeze(1).float()  # [F,1,L]

GFB = gfb_impulses().to(DEVICE)
ALPHA_LP = math.exp(-2*math.pi*LP_FC/SR)             # 1-pole decay

# ---------- simple loader ----------
def load_wav(path, sr=SR):
    x, s = sf.read(path)
    if x.ndim > 1: x = x.mean(1)
    if s != sr:    x = librosa.resample(y=x, orig_sr=s, target_sr=sr)
    return torch.from_numpy(x.astype("float32"))

# ---------- cochlea front-end (no adaptation) ----------
@torch.inference_mode()
def cochlea_lowpass(wave):
    # wave: [T] CPU tensor
    w = wave
    y = F.conv1d(w, GFB, padding=GFB.shape[-1]-1).clamp_(0)   # [1,F,T]
    # 1-pole LP
    for n in range(1, y.shape[-1]):
        y[..., n] = (1-ALPHA_LP)*y[..., n] + ALPHA_LP*y[..., n-1]
    return y.squeeze(0)  # [F,T]

# ---------- dataset in memory ----------
paths = sorted(glob.glob(os.path.join(AUDIO_DIR, "*.wav")))
wavs  = [load_wav(p).to(DEVICE).view(1,1,-1) for p in tqdm(paths[::2000], desc="pre-proc")]
data  = [cochlea_lowpass(wav) for wav in wavs]
Tmax  = max(t.shape[1] for t in data)
data  = torch.stack([F.pad(t, (0, Tmax-t.shape[1])) for t in data])  # [N,F,T]
print("dataset", data.shape)

class ConvCochlea(nn.Module):
    def __init__(self, n_filt=N_FILT, kf=63, kt=511):
        super().__init__()
        self.time_conv = nn.Conv1d(1, 1, kt, padding=kt//2, bias=False)
        self.freq_conv = nn.Conv1d(1, 1, kf, padding=kf//2, bias=False)
        self.mlp       = nn.Sequential(nn.Linear(80000, 80000))

    def forward(self, x):  # x: [B, F, T]

        B, F, T = x.shape
        
        # Apply temporal convolution across T (time axis)
        x_t = x.reshape(B*F, 1, T)
        x_t = self.time_conv(x_t)  # [B, F, T]
        x_t = x_t.reshape(B, F, T)

        x_t = torch.relu(x_t)

        # Downsample in time
        z_ds = x_t[..., ::N_FILT]                      # [B, F, T//N]
        T = T//N_FILT+1

        # Apply frequency convolution across F (freq axis)
        x_f = z_ds.transpose(1, 2)                # [B, T, F]
        x_f = x_f.reshape(B*T, 1, F)
        x_f = self.freq_conv(x_f)
        x_f = x_f.reshape(B, T, F).transpose(1, 2)  # back to [B, F, T]
        
        z_us = x_f.view(B, -1)

        y_hat = self.mlp(z_us)  # [B, F, T]
        return x_t, y_hat

model = ConvCochlea().to(DEVICE)
opt   = optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

# ---------- training loop ----------
for epoch in range(1, EPOCHS+1):
    perm = torch.randperm(len(data))
    tot_loss = 0
    for i in range(0, len(data), BATCH):
        idx  = perm[i:i+BATCH]
        batch = data[idx].to(DEVICE)                       # [B,F,T]
        wav_batch = wavs[idx]
        conv_out, recon = model(batch)
        # Match recon to batch size
        min_len = min(recon.shape[-1], batch.shape[-1])
        recon = recon[..., :min_len]
        batch = batch[..., :min_len]
        mse  = loss_fn(recon, wav_batch)
        l1   = (conv_out.abs().mean() /
                (conv_out.max().detach()+1e-8))
        loss = mse + LAMBDA_L1 * l1
        opt.zero_grad(); loss.backward(); opt.step()
        tot_loss += loss.item() * len(idx)
    print(f"epoch {epoch:2d} – loss {tot_loss/len(data):.6f}")

print("-- training done --")


pre-proc: 100%|███████████████████████████████████| 1/1 [00:00<00:00,  2.33it/s]


dataset torch.Size([1, 256, 80511])


RuntimeError: [enforce fail at alloc_cpu.cpp:119] err == 0. DefaultCPUAllocator: can't allocate memory: you tried to allocate 25600000000 bytes. Error code 12 (Cannot allocate memory)

In [2]:
import matplotlib.pyplot as plt

# Assume model is your trained model and .conv is your learned 2D convolution
kernel = model.time_conv.weight.detach().cpu().squeeze(0).squeeze(0)  # shape: [H, W]

plt.figure(figsize=(6,2))
plt.imshow(kernel[None], aspect='auto', cmap='bwr', origin='lower')
plt.colorbar(label='Weight')
plt.title("Learned 2D Kernel")
plt.xlabel("Time")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

NameError: name 'model' is not defined